# Question 2: Visa Category and Chargeability Area Comparison

**How do movement patterns differ across family-sponsored visa categories and chargeability areas?**

The analysis uses the full transformed Visa Bulletin dataset. Movement patterns are compared across visa categories and chargeability areas, with Final Action Dates and Dates for Filing evaluated separately where appropriate.

The main statistical approach for this question is the Chi-square test of independence to examine whether movement status is associated with visa category or chargeability area.


In [1]:
import pandas as pd

q2_df = pd.read_csv(
    "../data/processed/visa_bulletins_transformed_full.csv",
    parse_dates=["bulletin_date", "cutoff_date"]
)

q2_df.head()

,bulletin_date,visa_category,chargeability_area,table_type,raw_cutoff_value,cutoff_status,cutoff_date,days_behind_bulletin,movement_days,movement_status,months_stagnant,current_streak
0,2016-10-31,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,Date,2011-01-01,2130,NaN,NaN,0,0
1,2016-11-30,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,Date,2011-01-01,2160,0.0,No Movement,1,0
2,2016-12-31,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,Date,2011-01-01,2191,0.0,No Movement,2,0
3,2017-01-31,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,Date,2011-01-01,2222,0.0,No Movement,3,0
4,2017-02-28,F1,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN11,Date,2011-01-01,2250,0.0,No Movement,4,0


In [2]:
print("Shape:", q2_df.shape)
print("Visa categories:", q2_df["visa_category"].unique())
print("Chargeability areas:", q2_df["chargeability_area"].unique())
print("Table types:", q2_df["table_type"].unique())
print("Movement status counts:")
print(q2_df["movement_status"].value_counts(dropna=False))

Shape: (6000, 12)
Visa categories: <StringArray>
['F1', 'F2A', 'F2B', 'F3', 'F4']
Length: 5, dtype: str
Chargeability areas: <StringArray>
['All Chargeability Areas Except Those Listed',
                                       'China',
                                       'India',
                                      'Mexico',
                                 'Philippines']
Length: 5, dtype: str
Table types: <StringArray>
['Dates for Filing', 'Final Action']
Length: 2, dtype: str
Movement status counts:
movement_status
No Movement    3068
Advance        2852
NaN              50
Retrogress       30
Name: count, dtype: int64


In [7]:
#separate the data into two dataframes based on the table type
final_action_q2 = q2_df[
    q2_df["table_type"] == "Final Action"].copy()

dates_for_filing_q2 = q2_df[
    q2_df["table_type"] == "Dates for Filing"].copy()

print("Final Action Dates shape:", final_action_q2.shape)
print("Dates for Filing shape:", dates_for_filing_q2.shape)


Final Action Dates shape: (3000, 12)
Dates for Filing shape: (3000, 12)


In [8]:
# Remove rows where movement status cannot be calculated
# because there is no previous month for comparison
#drona does not effect the next month since values already exist for the next month, so we can drop these rows

final_action_q2 = final_action_q2.dropna(
    subset=["movement_status"]
)

dates_for_filing_q2 = dates_for_filing_q2.dropna(
    subset=["movement_status"]
)

In [9]:
print("Final Action rows:", len(final_action_q2))
print("Dates for Filing rows:", len(dates_for_filing_q2))

print("\nFinal Action movement counts:")
print(final_action_q2["movement_status"].value_counts())

print("\nDates for Filing movement counts:")
print(dates_for_filing_q2["movement_status"].value_counts())

Final Action rows: 2975
Dates for Filing rows: 2975

Final Action movement counts:
movement_status
Advance        1737
No Movement    1208
Retrogress       30
Name: count, dtype: int64

Dates for Filing movement counts:
movement_status
No Movement    1860
Advance        1115
Name: count, dtype: int64


In [10]:
# Compare movement status across visa categories for Final Action

final_action_category = pd.crosstab(
    final_action_q2["visa_category"],
    final_action_q2["movement_status"]
)

final_action_category

movement_status,Advance,No Movement,Retrogress
visa_category,,,
F1,303,284,8
F2A,458,126,11
F2B,323,271,1
F3,343,248,4
F4,310,279,6


In [11]:
filing_category = pd.crosstab(
    dates_for_filing_q2["visa_category"],
    dates_for_filing_q2["movement_status"]
)

filing_category

movement_status,Advance,No Movement
visa_category,,
F1,188,407
F2A,350,245
F2B,194,401
F3,193,402
F4,190,405


## Statistical Analysis

The previous steps prepared and validated the Q2 dataset by separating Final Action Dates and Dates for Filing and confirming the movement-status distributions.

The next step uses Chi-square tests of independence to examine whether movement status is associated with visa category or chargeability area. Final Action Dates and Dates for Filing are tested separately because their movement patterns differ.

### Final Action: Visa Category and Movement Status

In [1]:
%pip install scipy 
from scipy.stats import chi2_contingency


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
